In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Créer un plugin de transpileur

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les prérequis suivants.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
```
</details>
Créer un [plugin de transpileur](transpiler-plugins) est une excellente façon de partager ton code de transpilation avec la communauté Qiskit au sens large, permettant à d'autres utilisateurs de profiter des fonctionnalités que tu as développées. Merci de ton intérêt pour la contribution à la communauté Qiskit !

Avant de créer un plugin de transpileur, tu dois décider quel type de plugin convient à ta situation. Il existe trois types de plugins de transpileur :
- [**Plugin d'étape de transpileur**](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins). Choisis cette option si tu définis un gestionnaire de passes pouvant remplacer l'une des [6 étapes](transpiler-stages) d'un gestionnaire de passes par étapes prédéfini.
- [**Plugin de synthèse unitaire**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin). Choisis cette option si ton code de transpilation prend en entrée une matrice unitaire (représentée sous forme de tableau Numpy) et produit en sortie la description d'un circuit quantique implémentant cette unitaire.
- [**Plugin de synthèse haut niveau**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin). Choisis cette option si ton code de transpilation prend en entrée un « objet haut niveau » tel qu'un opérateur Clifford ou une fonction linéaire, et produit en sortie la description d'un circuit quantique implémentant cet objet. Les objets haut niveau sont représentés par des sous-classes de la classe [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation).

Une fois que tu as déterminé quel type de plugin créer, suis ces étapes pour le créer :

1. Crée une sous-classe de la classe abstraite de plugin appropriée :
   - [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin) pour un plugin d'étape de transpileur,
   - [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) pour un plugin de synthèse unitaire, et
   - [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) pour un plugin de synthèse haut niveau.
2. Expose la classe comme [point d'entrée setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) dans les métadonnées du package, généralement en modifiant le fichier `pyproject.toml`, `setup.cfg` ou `setup.py` de ton package Python.

Il n'y a pas de limite au nombre de plugins qu'un seul package peut définir, mais chaque plugin doit avoir un nom unique. Le SDK Qiskit lui-même inclut un certain nombre de plugins, dont les noms sont également réservés. Les noms réservés sont :

- Plugins d'étape de transpileur : voir [ce tableau](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#plugin-stages).
- Plugins de synthèse unitaire : `default`, `aqc`, `sk`
- Plugins de synthèse haut niveau :

| Classe d'opération | Nom de l'opération | Noms réservés |
| --- | --- | --- |
| [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford#clifford) | `clifford` | `default`, `ag`, `bm`, `greedy`, `layers`, `lnn` |
| [LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction#linearfunction) | `linear_function` | `default`, `kms`, `pmh` |
| [PermutationGate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.PermutationGate#permutationgate) | `permutation` | `default`, `kms`, `basic`, `acg`, `token_swapper` |

Dans les sections suivantes, nous montrons des exemples de ces étapes pour les différents types de plugins. Dans ces exemples, nous supposons que nous créons un package Python appelé `my_qiskit_plugin`. Pour des informations sur la création de packages Python, tu peux consulter [ce tutoriel](https://packaging.python.org/en/latest/tutorials/packaging-projects/) sur le site Python.
## Exemple : Créer un plugin d'étape de transpileur
Dans cet exemple, nous créons un plugin d'étape de transpileur pour l'étape `layout` (voir [Étapes du transpileur](transpiler-stages) pour une description des 6 étapes du pipeline de transpilation intégré de Qiskit).
Notre plugin exécute simplement [VF2Layout](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.VF2Layout) pour un nombre d'essais qui dépend du niveau d'optimisation demandé.

Tout d'abord, nous créons une sous-classe de [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin). Il y a une méthode à implémenter, appelée [`pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin#pass_manager). Cette méthode prend en entrée un [PassManagerConfig](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManagerConfig) et retourne le gestionnaire de passes que nous définissons. L'objet PassManagerConfig stocke des informations sur le backend cible, telles que sa carte de couplage et ses portes de base.

In [1]:
# This import is needed for python versions prior to 3.10
from __future__ import annotations

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import VF2Layout
from qiskit.transpiler.passmanager_config import PassManagerConfig
from qiskit.transpiler.preset_passmanagers import common
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePlugin,
)


class MyLayoutPlugin(PassManagerStagePlugin):
    def pass_manager(
        self,
        pass_manager_config: PassManagerConfig,
        optimization_level: int | None = None,
    ) -> PassManager:
        layout_pm = PassManager(
            [
                VF2Layout(
                    coupling_map=pass_manager_config.coupling_map,
                    properties=pass_manager_config.backend_properties,
                    max_trials=optimization_level * 10 + 1,
                    target=pass_manager_config.target,
                )
            ]
        )
        layout_pm += common.generate_embed_passmanager(
            pass_manager_config.coupling_map
        )
        return layout_pm

Ensuite, nous exposons le plugin en ajoutant un point d'entrée dans les métadonnées de notre package Python.
Ici, nous supposons que la classe définie est exposée dans un module appelé `my_qiskit_plugin`, par exemple en étant importée dans le fichier `__init__.py` du module `my_qiskit_plugin`.
Nous modifions le fichier `pyproject.toml`, `setup.cfg` ou `setup.py` de notre package (selon le type de fichier que tu as choisi pour stocker les métadonnées de ton projet Python) :

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.transpiler.layout"]
    "my_layout" = "my_qiskit_plugin:MyLayoutPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.transpiler.layout =
        my_layout = my_qiskit_plugin:MyLayoutPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.transpiler.layout': [
                'my_layout = my_qiskit_plugin:MyLayoutPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>
Consulte le [tableau des étapes de plugins de transpileur](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#stage-table) pour les points d'entrée et les attentes pour chaque étape de transpileur.

Pour vérifier que ton plugin est correctement détecté par Qiskit, installe ton package de plugin et suis les instructions dans [Plugins de transpileur](transpiler-plugins#list-available-transpiler-stage-plugins) pour lister les plugins installés, et assure-toi que ton plugin apparaît dans la liste :

In [2]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

If our example plugin were installed, then the name `my_layout` would appear in this list.

If you want to use a built-in transpiler stage as the starting point for your transpiler stage plugin, you can obtain the pass manager for a built-in transpiler stage using [PassManagerStagePluginManager](/docs/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). The following code cell shows how to do this to obtain the built-in optimization stage for optimization level 3.

In [3]:
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePluginManager,
)

# Initialize the plugin manager
plugin_manager = PassManagerStagePluginManager()

# Here we create a pass manager config to use as an example.
# Instead, you should use the pass manager config that you already received as input
# to the pass_manager method of your PassManagerStagePlugin.
pass_manager_config = PassManagerConfig()

# Obtain the desired built-in transpiler stage
optimization = plugin_manager.get_passmanager_stage(
    "optimization", "default", pass_manager_config, optimization_level=3
)

Si notre plugin d'exemple était installé, le nom `my_layout` apparaîtrait dans cette liste.

Si tu veux utiliser une étape de transpileur intégrée comme point de départ pour ton plugin d'étape de transpileur, tu peux obtenir le gestionnaire de passes d'une étape de transpileur intégrée à l'aide de [PassManagerStagePluginManager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). Le code suivant montre comment procéder pour obtenir l'étape d'optimisation intégrée au niveau d'optimisation 3.

In [4]:
import numpy as np
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit.dagcircuit import DAGCircuit
from qiskit.quantum_info import Operator
from qiskit.transpiler.passes import UnitarySynthesis
from qiskit.transpiler.passes.synthesis.plugin import UnitarySynthesisPlugin


class MyUnitarySynthesisPlugin(UnitarySynthesisPlugin):
    @property
    def supports_basis_gates(self):
        # Returns True if the plugin can target a list of basis gates
        return True

    @property
    def supports_coupling_map(self):
        # Returns True if the plugin can synthesize for a given coupling map
        return False

    @property
    def supports_natural_direction(self):
        # Returns True if the plugin supports a toggle for considering
        # directionality of 2-qubit gates
        return False

    @property
    def supports_pulse_optimize(self):
        # Returns True if the plugin can optimize pulses during synthesis
        return False

    @property
    def supports_gate_lengths(self):
        # Returns True if the plugin can accept information about gate lengths
        return False

    @property
    def supports_gate_errors(self):
        # Returns True if the plugin can accept information about gate errors
        return False

    @property
    def supports_gate_lengths_by_qubit(self):
        # Returns True if the plugin can accept information about gate lengths
        # (The format of the input differs from supports_gate_lengths)
        return False

    @property
    def supports_gate_errors_by_qubit(self):
        # Returns True if the plugin can accept information about gate errors
        # (The format of the input differs from supports_gate_errors)
        return False

    @property
    def min_qubits(self):
        # Returns the minimum number of qubits the plugin supports
        return None

    @property
    def max_qubits(self):
        # Returns the maximum number of qubits the plugin supports
        return None

    @property
    def supported_bases(self):
        # Returns a dictionary of supported bases for synthesis
        return None

    def run(self, unitary: np.ndarray, **options) -> DAGCircuit:
        basis_gates = options["basis_gates"]
        synth_pass = UnitarySynthesis(basis_gates, min_qubits=3)
        qubits = QuantumRegister(3)
        circuit = QuantumCircuit(qubits)
        circuit.append(Operator(unitary).to_instruction(), qubits)
        dag_circuit = synth_pass.run(circuit_to_dag(circuit))
        return dag_circuit

## Exemple : Créer un plugin de synthèse unitaire
Dans cet exemple, nous allons créer un plugin de synthèse unitaire qui utilise simplement la passe de transpilation intégrée [UnitarySynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.UnitarySynthesis#unitarysynthesis) pour synthétiser une porte. Bien entendu, ton propre plugin fera quelque chose de plus intéressant.

La classe [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) définit l'interface et le contrat pour les plugins de synthèse unitaire. La méthode principale est
[`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run),
qui prend en entrée un tableau Numpy stockant une matrice unitaire
et retourne un [DAGCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.dagcircuit.DAGCircuit) représentant le circuit synthétisé à partir de cette matrice unitaire.
En plus de la méthode `run`, plusieurs méthodes de propriété doivent être définies.
Voir [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) pour la documentation de toutes les propriétés requises.

Créons notre sous-classe UnitarySynthesisPlugin :

In [5]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

Si tu constates que les entrées disponibles pour la méthode [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run)
sont insuffisantes pour tes besoins, n'hésite pas à [ouvrir un ticket](https://github.com/Qiskit/qiskit/issues/new/choose) en expliquant tes exigences. Les modifications de l'interface du plugin, comme l'ajout d'entrées optionnelles supplémentaires, seront réalisées de manière rétrocompatible afin de ne pas nécessiter de changements de la part des plugins existants.

> **Note:** Toutes les méthodes préfixées par `supports_` sont réservées dans une classe dérivée de `UnitarySynthesisPlugin` dans le cadre de l'interface. Tu ne dois pas définir de méthodes `supports_*` personnalisées dans une sous-classe qui ne sont pas définies dans la classe abstraite.

Ensuite, nous exposons le plugin en ajoutant un point d'entrée dans les métadonnées de notre package Python.
Ici, nous supposons que la classe définie est exposée dans un module appelé `my_qiskit_plugin`, par exemple en étant importée dans le fichier `__init__.py` du module `my_qiskit_plugin`.
Nous modifions le fichier `pyproject.toml`, `setup.cfg` ou `setup.py` de notre package :

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.unitary_synthesis"]
    "my_unitary_synthesis" = "my_qiskit_plugin:MyUnitarySynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.unitary_synthesis =
        my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.unitary_synthesis': [
                'my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

Comme précédemment, si ton projet utilise `setup.cfg` ou `setup.py` au lieu de `pyproject.toml`, consulte la [documentation setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) pour adapter ces lignes à ta situation.

Pour vérifier que ton plugin est correctement détecté par Qiskit, installe ton package de plugin et suis les instructions dans [Plugins de transpileur](transpiler-plugins#list-available-unitary-synthesis-plugins) pour lister les plugins installés, et assure-toi que ton plugin apparaît dans la liste :

In [6]:
from qiskit.synthesis import synth_clifford_bm
from qiskit.transpiler.passes.synthesis.plugin import HighLevelSynthesisPlugin


class MyCliffordSynthesisPlugin(HighLevelSynthesisPlugin):
    def run(
        self,
        high_level_object,
        coupling_map=None,
        target=None,
        qubits=None,
        **options,
    ) -> QuantumCircuit:
        if high_level_object.num_qubits <= 3:
            return synth_clifford_bm(high_level_object)
        else:
            return None

This plugin synthesizes objects of type [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) that have
at most 3 qubits, using the `synth_clifford_bm` method.

Now, we expose the plugin by adding an entry point in our Python package metadata.
Here, we assume that the class we defined is exposed in a module called `my_qiskit_plugin`, for example by being imported in the `__init__.py` file of the `my_qiskit_plugin` module.
We edit the `pyproject.toml`, `setup.cfg`, or `setup.py` file of our package:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.synthesis"]
    "clifford.my_clifford_synthesis" = "my_qiskit_plugin:MyCliffordSynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.synthesis =
        clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.synthesis': [
                'clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

The `name` consists of two parts separated by a dot (`.`):
- The name of the type of [Operation](/docs/api/qiskit/qiskit.circuit.Operation) that the plugin synthesizes (in this case, `clifford`). Note that this string corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the Operation class, and not the name of the class itself.
- The name of the plugin (in this case, `special`).

As before, if your project uses `setup.cfg` or `setup.py` instead of `pyproject.toml`, see the [setuptools documentation](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) for how to adapt these lines for your situation.

To check that your plugin is successfully detected by Qiskit, install your plugin package and follow the instructions at [Transpiler plugins](transpiler-plugins#list-available-high-level-synthesis-plugins) for listing installed plugins, and ensure that your plugin appears in the list:

In [7]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn', 'rb_default']

Si notre plugin d'exemple était installé, le nom `my_unitary_synthesis` apparaîtrait dans cette liste.

Pour prendre en charge les plugins de synthèse unitaire qui exposent plusieurs options,
l'interface de plugin propose une option permettant aux utilisateurs de fournir un
dictionnaire de configuration libre. Celui-ci sera transmis à la méthode `run`
via l'argument nommé `options`. Si ton plugin dispose de ces options de configuration, tu dois les documenter clairement.

## Exemple : Créer un plugin de synthèse haut niveau

Dans cet exemple, nous allons créer un plugin de synthèse haut niveau qui utilise simplement la fonction intégrée [synth_clifford_bm](https://docs.quantum.ibm.com/api/qiskit/synthesis#synth_clifford_bm) pour synthétiser un opérateur Clifford.

La classe [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) définit l'interface et le contrat pour les plugins de synthèse haut niveau. La méthode principale est [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin#run).
L'argument positionnel `high_level_object` est une [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation) représentant l'objet « haut niveau » à synthétiser. Par exemple, il peut s'agir d'une
[LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) ou d'un
[Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford).
Les arguments nommés suivants sont présents :
- `target` spécifie le backend cible, permettant au plugin
d'accéder à toutes les informations spécifiques à la cible,
telles que la carte de couplage, l'ensemble de portes supportées, etc.
- `coupling_map` spécifie uniquement la carte de couplage, et n'est utilisé que lorsque `target` n'est pas spécifié.
- `qubits` spécifie la liste des qubits sur lesquels
l'objet haut niveau est défini, dans le cas où la synthèse est effectuée sur le circuit physique.
Une valeur de ``None`` indique que le placement n'a pas encore été choisi et que les qubits physiques dans la cible ou la carte de couplage sur lesquels cette opération s'applique n'ont pas encore été déterminés.
- `options`, un dictionnaire de configuration libre pour les options spécifiques au plugin. Si ton plugin dispose de ces options de configuration, tu dois les documenter clairement.

La méthode `run` retourne un [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit)
représentant le circuit synthétisé à partir de cet objet haut niveau.
Il est également possible de retourner `None`, indiquant que le plugin est incapable de synthétiser l'objet haut niveau donné.
La synthèse effective des objets haut niveau est réalisée par la passe de transpilation
[HighLevelSynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.HighLevelSynthesis).

En plus de la méthode `run`, plusieurs méthodes de propriété doivent être définies.
Voir [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) pour la documentation de toutes les propriétés requises.

Définissons notre sous-classe HighLevelSynthesisPlugin :